In [3]:
from pinecone import Pinecone, ServerlessSpec
pc = Pinecone(api_key="pcsk_5iqdag_4SZtMufSERfwzWbA389FCfPXRRN791goNysjdtsD2dxWFHZUVjdow2hZukyM7Hb")
import os
from openai import OpenAI
import pandas as pd
from time import time
import dotenv
dotenv.load_dotenv()

True

In [4]:
token= os.getenv("RUNPOD_TOKEN") 
open_ai_base_url = os.getenv("RUNPOD_EMBEDDING_URL")  
pinecone_api_key = os.getenv("PINECONE_API_KEY")

In [5]:

pc = Pinecone(api_key=pinecone_api_key)

client = OpenAI(
  api_key=token, 
  base_url=open_ai_base_url
)

# Try out embeddings


In [6]:
output = client.embeddings.create(input = ["helloo there"],model="BAAI/bge-small-en-v1.5")
embedings = output.data[0].embedding
print(embedings)


[-0.054519448429346085, -0.057440135627985, 0.08616019785404205, -0.06376828253269196, 0.017524108290672302, -0.012412910349667072, 0.05159876495599747, 0.052328936755657196, 0.02945023775100708, -0.022635307163000107, -0.013690710067749023, -0.04770451784133911, 0.029328543692827225, 0.03212753310799599, 0.05646657198667526, -0.009005445055663586, 0.012899691238999367, -0.054032668471336365, -0.1022239699959755, -0.02020140364766121, 0.030058713629841805, 0.05159876495599747, -0.0287200678139925, -0.035778388381004333, 0.0014375245664268732, -0.007575526367872953, 0.01947123184800148, 0.035048216581344604, -0.008214426226913929, -0.07350390404462814, -0.0010496211471036077, -0.012656300328671932, 0.04648756608366966, -0.0009963794145733118, 0.05159876495599747, 0.0001302518940065056, 0.05865708738565445, -0.025312600657343864, -0.07788492739200592, -0.005567555315792561, 0.06011742725968361, -0.024825820699334145, -0.0020840303041040897, -0.016307156533002853, 0.02300039306282997, -0.

In [7]:

len(embedings)

384

# Wrangle dataset


In [8]:

df=pd.read_json('products/products.jsonl',lines=True)

In [9]:
df.head(2)

,name,category,description,ingredients,price,rating,image_path
0,Cappuccino,Coffee,A rich and creamy cappuccino made with freshly...,"[Espresso, Steamed Milk, Milk Foam]",4.50,4.7,cappuccino.jpg
1,Jumbo Savory Scone,Bakery,"Deliciously flaky and buttery, this jumbo savo...","[Flour, Butter, Cheese, Herbs, Baking Powder, ...",3.25,4.3,SavoryScone.webp


In [10]:

df['text'] =  df['name']+" : "+df['description'] + \
                " -- Ingredients: " + df['ingredients'].astype(str) + \
                " -- Price: " + df['price'].astype(str) + \
                " -- rating: " + df['rating'].astype(str) 

In [11]:
df['text'].head()

0    Cappuccino : A rich and creamy cappuccino made...
1    Jumbo Savory Scone : Deliciously flaky and but...
2    Latte : Smooth and creamy, our latte combines ...
3    Chocolate Chip Biscotti : Crunchy and delightf...
4    Espresso shot : A bold shot of rich espresso, ...
Name: text, dtype: object

In [12]:

texts = df['text'].tolist()

In [13]:

with open('products/Merry\'s_way_about_us.txt') as f:
    Merry_way_about_section = f.read()
    
Merry_way_about_section = "Coffee shop Merry's Way about section: " + Merry_way_about_section
texts.append(Merry_way_about_section)

In [14]:

with open('products/menu_items_text.txt') as f:
    menue_items_text = f.read()
    
menue_items_text = "Menu Items: " + menue_items_text
texts.append(menue_items_text)

# Generate Embeddings